# BFCL V4 Smoke Test (Issue #5)

Validates the end-to-end evaluation pipeline on **V1 (zero-shot Qwen2.5-3B-Instruct base model)** by running the Non-Live category through the official bfcl-eval CLI via `src/evaluation/bfcl.py`.

**Acceptance for this PR:** a well-formed `results/v1/scores.json` matching our canonical schema, with at least `non_live` populated.

**Hardware:** T4 suffices for Non-Live (~700 entries) on Qwen2.5-3B at bf16. A100 recommended for full BFCL V4. Full V2/V3/V4/V5 runs are Issue #6, not this smoke test.

**Runtime estimate:** ~20-30 min on T4 for Non-Live; ~5-10 min on A100.

**Uses:** our fork `arnursabet/gorilla` on branch `qwen-2.5-3b-handler`, which registers `Qwen/Qwen2.5-3B-Instruct-FC -> QwenFCHandler` (upstream only registers Qwen3-*).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.chdir("/content/drive/MyDrive/info290-genai-spring2026/final-project")
!pwd

In [ ]:
# Install project deps plus bfcl-eval from our fork.
# The fork adds Qwen/Qwen2.5-3B-Instruct to the model registry.
!pip install -q -r requirements.txt

In [ ]:
# HuggingFace login (to pull Qwen2.5-3B-Instruct weights).
from huggingface_hub import login
login()

## Sanity checks

Before running the smoke test, confirm (1) bfcl-eval installed, (2) Qwen2.5-3B-Instruct-FC is registered in the model config, (3) the non_live test category is available.

In [ ]:
!bfcl --version
!bfcl test-categories | head -15
!bfcl models | grep -i qwen2.5-3b || echo 'ERROR: Qwen2.5-3B not registered. Check requirements.txt pins the correct fork branch.'

## Smoke test: V1 base Qwen, Non-Live

Generation config is milestone-locked in `configs/bfcl.yaml`: `temperature=0.0, top_p=1.0, max_tokens=512`.

First a dry run to show the subprocess commands, then the real execution.

In [ ]:
!python -m src.evaluation.bfcl --variant v1 --test-category non_live --dry-run

In [ ]:
# Real run. No adapter for V1 (zero-shot base).
!python -m src.evaluation.bfcl --variant v1 --test-category non_live

## Verify the canonical `scores.json`

Must conform to the schema in `docs/plans/2026-04-23-bfcl-v4-harness-plan.md`. The four non-Non-Live buckets should be `null` (not evaluated), and `non_live` should be a float in [0.0, 1.0]. Baseline expectation: Qwen2.5-3B base Non-Live Simple ~50-55% per published numbers.

In [ ]:
import json
from pathlib import Path

scores_path = Path("results/v1/scores.json")
assert scores_path.exists(), f"scores.json missing at {scores_path}"

data = json.loads(scores_path.read_text())

print("Variant:     ", data["variant"])
print("Model:       ", data["model"])
print("Adapter:     ", data["adapter"])
print("BFCL version:", data["bfcl_version"])
print("Gen config:  ", data["generation_config"])
print()
print("Top-level buckets:")
for bucket, score in data["top_level"].items():
    print(f"  {bucket:15s} = {score}")
print(f"\nOverall: {data['overall']:.4f}")
print(f"Weights: {data['weights']}")
print(f"\nSide metrics: {data['side_metrics']}")

# Schema assertions
assert set(data["top_level"].keys()) == {"non_live", "live", "hallucination", "multi_turn", "agentic"}
assert data["top_level"]["non_live"] is not None, "non_live should be populated"
assert 0.0 <= data["top_level"]["non_live"] <= 1.0
print("\nSCHEMA OK")

In [ ]:
# Inspect diagnostic dump: what did the model actually emit on the first 10 examples?
diag = Path("results/v1/diagnostics/first_10_generations.jsonl")
if diag.exists():
    for i, line in enumerate(open(diag)):
        if i >= 3:
            break
        entry = json.loads(line)
        print(f"Example {i}: id={entry.get('id')}")
        print(f"  raw:    {str(entry['raw_result'])[:200]}")
        print(f"  parsed: {entry['parsed_tool_calls']}")
        print()
else:
    print("No diagnostic file produced.")